# 중간고사 대체 실습과제 — 문제 5

## 시스템 다이내믹스: SaaS 성장 분석

| 항목 | 내용 |
|------|------|
| **관련 챕터** | Ch05. 시스템 다이내믹스 |
| **핵심 개념** | 피드백 루프(R/B), 스톡-플로우, 지연 |
| **배점** | 25점 |
| **제출** | 이 노트북(.ipynb)을 실행 결과 포함하여 제출 |

---

## 시나리오

B2B SaaS 스타트업 **CloudFlow**의 9개월 운영 데이터입니다.
초기 빠른 성장 후 정체가 시작되고 있습니다.

- 서버 용량: 동시접속 200명 기준
- 서버 증설: 발주 후 **3개월 지연**
- 월 구독료: 고객당 15만 원
- 이탈 사유(9월): 속도불만 45%, 경쟁사 20%, 기능부족 18%

In [1]:
# 환경설정
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)

In [2]:
# 데이터 로드
data = pd.DataFrame({
    '월': [1, 2, 3, 4, 5, 6, 7, 8, 9],
    '고객수': [100, 115, 135, 158, 176, 189, 196, 198, 195],
    '신규가입': [18, 20, 28, 35, 32, 28, 25, 22, 19],
    '이탈': [3, 5, 8, 12, 14, 15, 18, 20, 22],
    '서버응답시간_ms': [120, 130, 180, 250, 380, 520, 650, 720, 780],
    'CS문의건수': [25, 32, 48, 72, 95, 130, 158, 175, 190]
})

print(data.to_string(index=False))

 월  고객수  신규가입  이탈  서버응답시간_ms  CS문의건수
 1  100    18   3        120      25
 2  115    20   5        130      32
 3  135    28   8        180      48
 4  158    35  12        250      72
 5  176    32  14        380      95
 6  189    28  15        520     130
 7  196    25  18        650     158
 8  198    22  20        720     175
 9  195    19  22        780     190


---

## 과제 1. 데이터 탐색과 피드백 루프 식별 (8점)

### 요구사항

1. **(3점)** 6개 변수의 시계열 추이를 **서브플롯(2×3)** 으로 시각화하세요.

2. **(2점)** **서버응답시간**과 **이탈** 간 상관계수를 계산하여 `print()`로 출력하세요.

3. **(3점)** 이 시스템에서 관찰되는 **강화 루프(R) 1개**와 **균형 루프(B) 1개**를 찾아 아래 형식으로 서술하세요.

```
R1 (이름): 변수A → (+) 변수B → (+) 변수C → (+) 변수A
B1 (이름): 변수A → (+) 변수B → (-) 변수C → ... → 변수A
```

In [7]:
# ✏️ 과제 1-1: 6개 변수 서브플롯 시각화
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# 1. 데이터 로드
data = pd.DataFrame({
    '월': [1, 2, 3, 4, 5, 6, 7, 8, 9],
    '고객수': [100, 115, 135, 158, 176, 189, 196, 198, 195],
    '신규가입': [18, 20, 28, 35, 32, 28, 25, 22, 19],
    '이탈': [3, 5, 8, 12, 14, 15, 18, 20, 22],
    '서버응답시간_ms': [120, 130, 180, 250, 380, 520, 650, 720, 780],
    'CS문의건수': [25, 32, 48, 72, 95, 130, 158, 175, 190]
})
# '월' 데이터를 문자열로 변환하여 x축 가독성 확보
data['월_str'] = data['월'].astype(str) + '월'

# 2. 2x3 서브플롯 생성
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=("총 고객 수", "신규 가입", "이탈 고객", "서버 응답 시간(ms)", "CS 문의 건수", "고객당 CS 비율"),
    vertical_spacing=0.15,
    horizontal_spacing=0.1
)

# 공통 마커/선 스타일 설정
trace_style = dict(
    mode='lines+markers',
    line=dict(color='black', width=2),
    marker=dict(color='white', size=8, line=dict(color='black', width=2))
)

# 각 그래프 추가
# (1, 1) 총 고객 수
fig.add_trace(go.Scatter(x=data['월_str'], y=data['고객수'], name="고객수", **trace_style), row=1, col=1)
# (1, 2) 신규 가입
fig.add_trace(go.Scatter(x=data['월_str'], y=data['신규가입'], name="신규가입", **trace_style), row=1, col=2)
# (1, 3) 이탈 고객
fig.add_trace(go.Scatter(x=data['월_str'], y=data['이탈'], name="이탈", **trace_style), row=1, col=3)
# (2, 1) 서버 응답 시간
fig.add_trace(go.Scatter(x=data['월_str'], y=data['서버응답시간_ms'], name="응답시간", **trace_style), row=2, col=1)
# (2, 2) CS 문의 건수
fig.add_trace(go.Scatter(x=data['월_str'], y=data['CS문의건수'], name="CS건수", **trace_style), row=2, col=2)
# (2, 3) 고객당 CS 비율 (파생 지표)
fig.add_trace(go.Scatter(x=data['월_str'], y=data['CS문의건수']/data['고객수'], name="CS비율", **trace_style), row=2, col=3)

# 3. 레이아웃 및 공통 규칙 적용
fig.update_layout(
    title_text="<b>CloudFlow 주요 운영 지표 시계열 분석 (1~9월)</b>",
    title_font=dict(size=20, color='black'),
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(color='black'),
    showlegend=False,
    height=700,
    width=1000
)

# 모든 축에 대해 그리드 및 텍스트 색상 설정
fig.update_xaxes(showgrid=True, gridcolor='lightgrey', linecolor='black', tickfont=dict(color='black'))
fig.update_yaxes(showgrid=True, gridcolor='lightgrey', linecolor='black', tickfont=dict(color='black'))

fig.show()

In [8]:
# ✏️ 과제 1-2: 서버응답시간 vs 이탈 상관계수
import pandas as pd

# 1. 데이터 로드
data = pd.DataFrame({
    '월': [1, 2, 3, 4, 5, 6, 7, 8, 9],
    '고객수': [100, 115, 135, 158, 176, 189, 196, 198, 195],
    '신규가입': [18, 20, 28, 35, 32, 28, 25, 22, 19],
    '이탈': [3, 5, 8, 12, 14, 15, 18, 20, 22],
    '서버응답시간_ms': [120, 130, 180, 250, 380, 520, 650, 720, 780],
    'CS문의건수': [25, 32, 48, 72, 95, 130, 158, 175, 190]
})

# 2. 상관계수 계산
correlation = data['서버응답시간_ms'].corr(data['이탈'])

# 3. 결과 출력
print(f"서버 응답 시간과 이탈 간의 상관계수: {correlation:.4f}")

서버 응답 시간과 이탈 간의 상관계수: 0.9664


### ✏️ 과제 1-3: R/B 루프 서술 (이 셀에 직접 작성)

**R1 ():**

(R1 (성장의 가속 루프): 고객 유입 및 구전 효과

신규 가입 → (+) 고객 수 → (+) 서비스 인지도/매출 → (+) 신규 가입)

**B1 ( ):**

B1 (성장의 한계 균형 루프): 서버 과부하에 따른 이탈

고객 수 → (+) 서버 부하(응답 시간) → (+) 고객 불만(CS) → (+) 이탈 → (-) 고객 수



---

## 과제 2. 스톡-플로우 시뮬레이션 (10점)

CloudFlow의 고객 수를 **스톡-플로우 모델**로 36개월 시뮬레이션하세요.

### 모델 구조

```
  신규가입 ──▶ [ 고객 수 (Stock) ] ──▶ 이탈
                     │
              부하율 = 고객수 / 서버용량
                     │
              품질 (부하율 > 0.8이면 저하)
                     │
           ┌─────────┴─────────┐
           ▼                   ▼
     구전 감소              이탈 증가
```

### 사용할 파라미터

| 파라미터 | 값 |
|---------|-----|
| 초기 고객 | 100 |
| 초기 서버 용량 | 200 |
| 구전율 | 0.035 (고객당 월 신규 유입) |
| 기본 신규 유입 | 13명/월 |
| 기본 이탈률 | 0.02 |
| 품질 민감도 | 0.04 |
| 부하 임계치 | 0.8 |
| 투자 지연 | 3개월 |
| 투자당 추가 용량 | 50 |

### 요구사항

1. **(5점)** 위 파라미터로 36개월 시뮬레이션 함수를 구현하세요.
   - 투자 전략 인자를 받도록 설계: `none`(투자 없음) / `proactive`(6개월마다 투자)

2. **(5점)** 두 전략(`none` vs `proactive`)의 **고객 수 추이**를 하나의 라인 차트로 비교하고,
   36개월 후 고객 수 차이를 `print()`로 출력하세요.

In [11]:
# ✏️ 과제 2-1: 시뮬레이션 함수 구현
import pandas as pd
import plotly.graph_objects as go

def run_cloudflow_sim(strategy='none'):
    # 초기값 설정
    months = 36
    customer = 100
    server_capacity = 200
    
    # 파라미터 정의
    wom_rate = 0.035        # 구전율
    base_inflow = 13        # 기본 신규 유입
    base_churn = 0.02       # 기본 이탈률
    sensitivity = 0.04      # 품질 민감도
    threshold = 0.8         # 부하 임계치
    inv_delay = 3           # 투자 지연 (3개월)
    inv_unit = 50           # 투자당 추가 용량
    
    history = []
    pending_investments = [] # (용량 반영 예정 월, 증설량)

    for m in range(1, months + 1):
        # [1] 투자 반영 (지연된 투자가 완료되는 시점)
        for inv in pending_investments[:]:
            if inv[0] == m:
                server_capacity += inv[1]
                pending_investments.remove(inv)
        
        # [2] 투자 결정 (Proactive 전략: 6개월마다 발주)
        if strategy == 'proactive' and m % 6 == 0:
            pending_investments.append((m + inv_delay, inv_unit))
            
        # [3] 시스템 상태 계산
        load_factor = customer / server_capacity
        quality_loss = max(0, load_factor - threshold) # 0.8 초과분 계산
        
        # [4] 플로우(Flow) 계산
        # 신규 유입: 품질 저하 시 구전 효과 감소
        current_wom = max(0, wom_rate - (quality_loss * sensitivity))
        inflow = base_inflow + (customer * current_wom)
        
        # 이탈: 품질 저하 시 이탈률 증가
        current_churn_rate = base_churn + (quality_loss * sensitivity)
        outflow = customer * current_churn_rate
        
        # [5] 스톡(Stock) 업데이트
        customer = max(0, customer + inflow - outflow)
        
        history.append({
            'Month': m,
            'Customer': round(customer),
            'Capacity': server_capacity,
            'Load': round(load_factor, 2)
        })
        
    return pd.DataFrame(history)

# 시뮬레이션 실행
df_none = run_cloudflow_sim(strategy='none')
df_proactive = run_cloudflow_sim(strategy='proactive')
        
       

In [12]:
# ✏️ 과제 2-2: none vs proactive 비교 라인 차트 + 결과 print()
import plotly.graph_objects as go

# 차트 생성
fig = go.Figure()

# 1. 투자 없음 (none) - 회색 점선으로 '정체' 표현
fig.add_trace(go.Scatter(
    x=df_none['Month'], 
    y=df_none['Customer'],
    name='전략: none (투자 없음)',
    mode='lines+markers',
    line=dict(color='grey', width=2, dash='dot'),
    marker=dict(color='white', size=6, line=dict(color='black', width=1))
))

# 2. 선제적 투자 (proactive) - 검정 실선으로 '성장' 표현
fig.add_trace(go.Scatter(
    x=df_proactive['Month'], 
    y=df_proactive['Customer'],
    name='전략: proactive (6개월 주기 투자)',
    mode='lines+markers',
    line=dict(color='black', width=3),
    marker=dict(color='white', size=8, line=dict(color='black', width=2))
))

# 레이아웃 및 공통 규칙 적용
fig.update_layout(
    title="<b>CloudFlow 전략별 고객 수 시뮬레이션 (36개월)</b>",
    title_font=dict(size=20, color='black'),
    xaxis=dict(title="개월 (Month)", showgrid=True, gridcolor='lightgrey', tickfont=dict(color='black')),
    yaxis=dict(title="고객 수 (Person)", showgrid=True, gridcolor='lightgrey', tickfont=dict(color='black')),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    legend=dict(x=0.02, y=0.98, bordercolor='black', borderwidth=1),
    width=1000,
    height=600
)

fig.show()

### ✏️ 과제 2 해석 (이 셀에 직접 타이핑)

- 36개월 후 두 전략의 고객 수 차이는 몇 명인가?
총 283명이다
- 선제적 투자 전략이 효과적인 이유를 **부하율과 품질 저하** 관점에서 1~2문장으로 설명하세요.

---

## 과제 3. 전략 제안 (7점)

### 요구사항

1. **(3점)** 과제 1~2 결과를 바탕으로, 성장 정체의 **시스템적 원인**을 피드백 루프와 지연 관점에서 2~3문장으로 서술하세요.

2. **(4점)** CEO에게 **추천 전략 1개**를 제안하세요:
   - 어떤 **레버리지 포인트**에 개입하는 전략인지
   - 시뮬레이션 결과에서 어떤 근거로 선택했는지
   - 3문장 이내로 작성

### ✏️ 과제 3-1: 시스템적 원인 분석

(여기에 2~3문장 작성)

### ✏️ 과제 3-2: 전략 제안

(여기에 3문장 이내 작성)

---

## 채점 기준

| 과제 | 배점 | 채점 포인트 |
|------|------|------------|
| 과제 1. 데이터 탐색 | 8점 | 서브플롯(3) + 상관계수(2) + R/B 루프(3) |
| 과제 2. 시뮬레이션 | 10점 | 함수 구현(5) + 비교 시각화(5) |
| 과제 3. 전략 제안 | 7점 | 원인 분석(3) + 전략 제안(4) |
| **합계** | **25점** | |

---

## 참고: 예상 정답값

- 서버응답시간 ↔ 이탈 상관계수: **약 0.98** (매우 강한 양의 상관)

**시뮬레이션 36개월 후 (근사치):**

| 전략 | 36개월 후 고객 |
|------|---------------|
| 투자 없음 (none) | ~287명 |
| 선제적 투자 (proactive) | ~478명 |

- 선제적 투자 시 약 190명 더 확보 가능
- 핵심 루프: 고객↑ → 부하↑ → 품질↓ → 이탈↑, 구전↓ → 고객 정체 (균형 루프)
- 레버리지 포인트: 투자 지연(3개월)을 줄이거나, 선제적으로 용량을 확보하는 것이 핵심